## Part 4: Models, metrics and XAI (P4)

This part trains and compares the models on the same modeling table, chooses the metrics and the threshold without touching the test set, and explains the models.

Sections: 4.1) shared evaluation function `train_eval()`, 4.2) draft data, 4.3) sanity check of the metrics.

In [ ]:
# LOCAL SETUP (removed automatically when the notebooks are merged)
import sys
sys.path.append("../src")
from f1_clean import load_clean, SEED

In [ ]:
clean = load_clean()
base = clean["model_base"]      # pre-race columns + the target `podium`
outcomes = clean["outcomes"]    # post-race columns (status, points...). Never features
tables = clean["tables"]        # all other cleaned tables

### 4.1 The evaluation function `train_eval()`

One function scores every model the same way, so all results (our models and P3's ablation) are directly comparable.

**Why these metrics?** The brief asks for ROC-AUC, PR-AUC and F1 for every model; we also report precision and recall, because F1 alone hides which of the two is weak.
* **ROC-AUC:** how well the model separates podium from non-podium overall (uses probabilities, no threshold).
* **PR-AUC:** how well the model ranks the real podium finishers at the top (uses probabilities). Computed as *average precision*, the standard summary of the precision-recall curve. It is more informative than ROC-AUC when the positive class is rare, and podiums are rare (about 14% of rows).
* **F1, precision, recall:** quality of the yes/no decision at a chosen probability threshold. The threshold is 0.5 for now; it will be tuned on validation in a later step.

**Leakage safety:** the model is fitted on the training years only. The test set is deliberately *not* an input: it is scored once, at the very end, with `evaluate()`.

The same code is in `src/f1_eval.py`, so P3 can import it for the ablation.

In [ ]:
import pandas as pd
from sklearn.metrics import (average_precision_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def evaluate(model, X, y, threshold=0.5):
    """Score an already-fitted model on one split and return the 5 metrics.
    ROC-AUC and PR-AUC use the probabilities (no threshold needed);
    F1, precision and recall use the yes/no prediction at `threshold`."""
    proba = model.predict_proba(X)[:, 1]          # predicted probability of a podium
    pred = (proba >= threshold).astype(int)       # 1 = "podium" at this threshold
    return {
        "ROC-AUC": roc_auc_score(y, proba),
        "PR-AUC": average_precision_score(y, proba),
        "F1": f1_score(y, pred, zero_division=0),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
    }


def train_eval(model, X_train, y_train, X_val, y_val, threshold=0.5, name="model"):
    """Fit `model` on the training years only, then score it on train and validation.
    The test set is not used here on purpose: it is scored once, at the end, with evaluate()."""
    model.fit(X_train, y_train)
    rows = [
        {"model": name, "split": "train", **evaluate(model, X_train, y_train, threshold)},
        {"model": name, "split": "validation", **evaluate(model, X_val, y_val, threshold)},
    ]
    return pd.DataFrame(rows).round(4)

### 4.2 Draft data (temporary)

P3's modeling table is not ready yet (draft on Oct 8, frozen on Oct 10). To test the function now, we use `model_base` with the team's time split (train <= 2019, validation 2020-2021) and four pre-race columns. This cell will be replaced by P3's table; the test years (>= 2022) are not touched here.

In [ ]:
DRAFT_FEATURES = ["grid", "pitlane_start", "has_qualifying", "qual_position"]

train = base[base["year"] <= 2019]
val = base[base["year"].between(2020, 2021)]

X_train, y_train = train[DRAFT_FEATURES].astype(float), train["podium"]
X_val, y_val = val[DRAFT_FEATURES].astype(float), val["podium"]

print(f"train: {len(X_train)} rows, podium rate {y_train.mean():.3f}")
print(f"validation: {len(X_val)} rows, podium rate {y_val.mean():.3f}")

### 4.3 Sanity check of the metrics

Before trusting the function, we test it on a model that knows nothing: `DummyClassifier` always predicts the training podium rate. A correct implementation must give **ROC-AUC = 0.5** (no separation) and **PR-AUC = the podium rate** (no better than random ranking), and **F1 = 0** at threshold 0.5 (it never says "podium"). We also print its accuracy, to show why accuracy is not one of our metrics.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="prior", random_state=SEED)
dummy_results = train_eval(dummy, X_train, y_train, X_val, y_val, name="dummy (always the base rate)")
display(dummy_results)

val_row = dummy_results[dummy_results["split"] == "validation"].iloc[0]
assert abs(val_row["ROC-AUC"] - 0.5) < 1e-3
assert abs(val_row["PR-AUC"] - y_val.mean()) < 1e-3
assert val_row["F1"] == 0
print(f"Dummy accuracy on validation: {accuracy_score(y_val, dummy.predict(X_val)):.1%} "
      f"(looks good, but it finds 0 podiums)")

**What we see.** The dummy model gets ROC-AUC 0.5, PR-AUC equal to the podium rate and F1 = 0, exactly as expected, so the function computes the metrics correctly. Its accuracy is still about 85%, although it never finds a single podium. That is why we do not use accuracy, and why **PR-AUC is the natural primary metric** for this rare positive class. These dummy numbers are also the floor every real model must beat.

**Next step:** the baseline (logistic regression plus a `grid <= 3` rule) on P3's modeling table.